# Credit Card Fraud Detection - PCA Encrypted Dataset

**Dataset Overview & Anomaly Detection:**
This dataset contains credit card transactions, and the primary goal is anomaly detection (identifying fraudulent transactions among legitimate ones).
Due to confidentiality and privacy concerns, the dataset is a **PCA (Principal Component Analysis) encrypted dataset**. 
- The original features are not provided.
- Features `V1`, `V2`, ... `V28` are the principal components obtained through the PCA transformation.
- The only features that have not been transformed are `Time` (seconds elapsed between each transaction and the first transaction) and `Amount` (transaction amount).
- The `Class` feature is the target variable, where `1` represents a fraudulent transaction and `0` represents a legitimate transaction.

This severe class imbalance requires specialized handling and evaluation metrics.

# GPU Accelerated Mathematical Model
This notebook has been optimized:
1. **SMOTE** is fully vectorized on the CPU, skipping 90 million redundant calculations to finish in ~1 second.
2. **Cost-Sensitive Logistic Regression** uses CuPy to run Gradient Descent on your RTX 4050 GPU (with a NumPy fallback if CuPy isn't installed).


In [ ]:
import pandas as pd
import numpy as np
import random
import time

try:
    import cupy as xp
    # Test if CuPy/CUDA is fully functional
    _ = xp.dot(xp.array([1.0]), xp.array([1.0]))
    print("CuPy successfully imported and functional! GPU will be used.")
except Exception as e:
    import numpy as xp
    print(f"CuPy found but not functional ({e}). Falling back to NumPy (CPU).")


In [ ]:
df = pd.read_csv("../dataset_statistics_and_feature_selection/creditcard.csv")

# 1. Define the exact 12 features chosen during Feature Selection
final_features = ['V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V2', 'V3', 'V4', 'V7', 'V9']

# 2. Extract only those 12 columns as your input matrix (X)
X = df[final_features].values

# 3. Extract the target column (y)
y = df['Class'].values


In [ ]:
def vectorized_smote(minority_data, num_synthetic_samples, k=5):
    """
    Step B: Generates synthetic fraud data points using vectorized distance calculations.
    Runs in <1 second on CPU.
    """
    minority_data = np.array(minority_data)
    num_minority = len(minority_data)
    
    print(f"Pre-computing nearest neighbors for {num_minority} fraud points...")
    
    # 1. Pre-compute distances between all minority points
    diff = minority_data[:, np.newaxis, :] - minority_data[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff**2, axis=-1))
    
    # 2. Find the k nearest neighbors for EACH point
    # We use argsort. The closest point is always itself (distance 0), so we take 1 to k+1
    nearest_neighbor_indices = np.argsort(distances, axis=1)[:, 1:k+1]
    
    synthetic_data = []
    
    print(f"Generating {num_synthetic_samples} synthetic samples...")
    # 3. Quickly generate the samples using the pre-computed neighbors
    for _ in range(num_synthetic_samples):
        # Pick a random fraud transaction
        core_idx = random.randint(0, num_minority - 1)
        core_point = minority_data[core_idx]
        
        # Pick a random neighbor from its pre-computed list
        neighbor_idx = random.choice(nearest_neighbor_indices[core_idx])
        chosen_neighbor = minority_data[neighbor_idx]
        
        # Interpolation math
        lambda_val = random.random()
        synthetic_point = core_point + lambda_val * (chosen_neighbor - core_point)
        synthetic_data.append(synthetic_point)
        
    return np.array(synthetic_data)


In [ ]:
class GPULogisticRegression:
    def __init__(self, learning_rate=0.01, epochs=1000, weight_fraud=1.0, weight_genuine=1.0):
        self.lr = learning_rate
        self.epochs = epochs
        self.w1 = weight_fraud
        self.w0 = weight_genuine
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        z = xp.clip(z, -250, 250)
        return 1 / (1 + xp.exp(-z))

    def fit(self, X, y):
        print("Transferring data to GPU (if available)...")
        X_gpu = xp.array(X)
        y_gpu = xp.array(y)
        
        num_samples, num_features = X_gpu.shape
        self.weights = xp.zeros(num_features)
        self.bias = 0

        print(f"Running Gradient Descent for {self.epochs} epochs...")
        start_time = time.time()
        
        for _ in range(self.epochs):
            linear_model = xp.dot(X_gpu, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)
            
            raw_error = y_predicted - y_gpu
            weighted_error = xp.where(y_gpu == 1, raw_error * self.w1, raw_error * self.w0)

            dw = (1 / num_samples) * xp.dot(X_gpu.T, weighted_error)
            db = (1 / num_samples) * xp.sum(weighted_error)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        end_time = time.time()
        print(f"Training completed in {end_time - start_time:.2f} seconds.")
            
        # Keep weights on CPU for future predictions if needed
        self.weights = xp.asnumpy(self.weights) if hasattr(xp, 'asnumpy') else self.weights
        self.bias = float(self.bias)

    def predict(self, X, threshold=0.5):
        X_gpu = xp.array(X)
        weights_gpu = xp.array(self.weights)
        
        linear_model = xp.dot(X_gpu, weights_gpu) + self.bias
        y_predicted = self._sigmoid(linear_model)
        
        y_pred_cpu = xp.asnumpy(y_predicted) if hasattr(xp, 'asnumpy') else y_predicted
        return np.array([1 if i > threshold else 0 for i in y_pred_cpu])


In [ ]:
def custom_train_test_split(X, y, test_size=0.2, random_seed=42):
    np.random.seed(random_seed)
    
    fraud_indices = np.where(y == 1)[0]
    genuine_indices = np.where(y == 0)[0]
    
    np.random.shuffle(fraud_indices)
    np.random.shuffle(genuine_indices)
    
    fraud_split = int(len(fraud_indices) * (1 - test_size))
    genuine_split = int(len(genuine_indices) * (1 - test_size))
    
    train_fraud, test_fraud = fraud_indices[:fraud_split], fraud_indices[fraud_split:]
    train_genuine, test_genuine = genuine_indices[:genuine_split], genuine_indices[genuine_split:]
    
    train_idx = np.concatenate([train_fraud, train_genuine])
    test_idx = np.concatenate([test_fraud, test_genuine])
    
    np.random.shuffle(train_idx)
    np.random.shuffle(test_idx)
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def evaluate_model(y_true, y_predicted):
    TP = np.sum((y_true == 1) & (y_predicted == 1))
    TN = np.sum((y_true == 0) & (y_predicted == 0))
    FP = np.sum((y_true == 0) & (y_predicted == 1))
    FN = np.sum((y_true == 1) & (y_predicted == 0))

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1_score = 2 * (precision * recall) / (precision + recall + 1e-9)

    print("\n--- CUSTOM EVALUATION METRICS ---")
    print(f"True Positives (Caught Fraud): {TP}")
    print(f"False Negatives (Missed Fraud): {FN}")
    print(f"False Positives (False Alarms): {FP}")
    print(f"True Negatives (Legitimate Passed): {TN}")
    print("-" * 30)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1_score:.4f}")

    return precision, recall, f1_score


In [ ]:
# 1. Stratified Split
print("Splitting data...")
X_train, X_test, y_train, y_test = custom_train_test_split(X, y, test_size=0.2)

# 2. Extract minority data
minority_mask = (y_train == 1)
X_minority = X_train[minority_mask]

num_genuine = np.sum(y_train == 0)
num_fraud = np.sum(y_train == 1)
num_synthetic_to_generate = num_genuine - num_fraud

# 3. Fast Vectorized SMOTE
print("Starting SMOTE...")
smote_start = time.time()
synthetic_X = vectorized_smote(X_minority, num_synthetic_samples=num_synthetic_to_generate, k=5)
smote_end = time.time()
print(f"SMOTE completed in {smote_end - smote_start:.2f} seconds.")

synthetic_y = np.ones(len(synthetic_X))

# 4. Combine data
X_train_balanced = np.vstack((X_train, synthetic_X))
y_train_balanced = np.concatenate((y_train, synthetic_y))

# 5. Initialize GPU Model
# Standard LR on balanced data (weight_fraud=1.0)
model = GPULogisticRegression(learning_rate=0.01, epochs=1000, weight_fraud=1.0, weight_genuine=1.0)

# 6. Train the model
model.fit(X_train_balanced, y_train_balanced)

# 7. Predictions & Evaluation
print("Making predictions...")
y_predicted = model.predict(X_test, threshold=0.5)

precision, recall, f1 = evaluate_model(y_test, y_predicted)
